
# Spitzer IRAC AGN Wedge Diagram

Color-color diagram in Spitzer IRAC bands (3.6, 4.5, 5.8, 8.0 μm) showing
the Lacy+2007 / Donley+2012 AGN selection wedge. Population of 50 star-forming
galaxies (z=0–2) are plotted as blue cloud; 10 AGN with varying bolometric
luminosity cluster inside the wedge (red region) demonstrating the diagnostic
power of mid-infrared colors for AGN identification.

References: Lacy et al. (2007) ApJ 669, 54–64; Donley et al. (2012)
ApJ 748, 142.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()

irac_bands = ["irac_36", "irac_45", "irac_58", "irac_80"]
obs_irac = tengri.Observation(photometry=tengri.Photometry.from_names(irac_bands))


# --- Helper: draw SF galaxy with randomized params from prior ---
def sample_sf_galaxy(ssp, key, redshift):
    """
    Sample a star-forming galaxy with random params drawn from broad priors.
    Returns photometry in IRAC bands.
    """
    subkey1, _ = jr.split(key)

    model = tengri.SEDModel.build(
        ssp_data=ssp,
        observation=tengri.Observation(photometry=obs_irac.photometry),
        sfh={
            "type": "tsnorm",
            "log_total_mass": 10.0,
            "peak_lbt_gyr": tengri.Uniform(0.5, 10.0),
            "width_gyr": tengri.Uniform(0.3, 4.0),
            "skew": tengri.Uniform(-2.0, 2.0),
            "trunc": tengri.Uniform(1.0, 8.0),
        },
        dust={
            "type": "two_component",
            "law_bc": "calzetti",
            "tau_bc": tengri.Uniform(0.0, 0.8),
            "tau_diff": tengri.Uniform(0.0, 0.5),
            "slope": tengri.Fixed(-0.7),
        },
        redshift=tengri.Fixed(redshift),
    )

    params = model.spec.sample(subkey1)
    phot = model.predict_photometry(params)

    return np.asarray(phot)


# --- Helper: composite (host + AGN) photometry at varying AGN fraction.
# IRAC colors are *shape* quantities, so varying the AGN bolometric
# luminosity alone leaves them unchanged. What actually moves a galaxy
# across the wedge is the relative AGN-to-host contribution in the
# mid-IR — that is what we sweep here.
def composite_photometry(ssp, agn_lum_ratio, redshift):
    model = tengri.SEDModel.build(
        ssp_data=ssp,
        observation=tengri.Observation(photometry=obs_irac.photometry),
        sfh={
            "type": "dpl",
            "all_params": tengri.FIXED,
            "log_total_mass": 10.0,
            "tau_gyr": 1.5,
            "alpha": 2.0,
            "beta": 2.5,
        },
        dust={
            "type": "two_component",
            "all_params": tengri.FIXED,
            "tau_bc": 0.4,
            "tau_diff": 0.2,
        },
        agn={
            "type": "composable",
            "all_params": tengri.FIXED,
            "agn_lum_ratio": agn_lum_ratio,  # 0 = pure host, 1 = AGN-dominated
            "agn_log_lbol": 12.5,
            "disc": {"type": "multicolor", "all_params": tengri.FIXED},
            "torus": {"type": "skirtor", "all_params": tengri.FIXED},
            "nlr": {"type": "analytic", "all_params": tengri.FIXED},
            "blr": {"type": "none", "all_params": tengri.FIXED},
        },
        redshift=tengri.Fixed(redshift),
    )

    params = model.spec.sample(jax.random.PRNGKey(0))
    phot = model.predict_photometry(params)

    return np.asarray(phot)


# --- Generate star-forming galaxy population (z=0–2) ---
n_sf = 50
redshifts_sf = np.linspace(0.0, 2.0, n_sf)

sf_phot = []
for i, z in enumerate(redshifts_sf):
    key = jr.fold_in(jr.PRNGKey(42), i)
    flux = sample_sf_galaxy(ssp, key, z)
    sf_phot.append(flux)

sf_phot = np.array(sf_phot)

# IRAC channel indices: ch1=0 (3.6), ch2=1 (4.5), ch3=2 (5.8), ch4=3 (8.0)
# Lacy/Donley colors: log(f_5.8/f_3.6) vs log(f_8.0/f_4.5)
with np.errstate(divide="ignore", invalid="ignore"):
    sf_color_x = np.log10(sf_phot[:, 2] / sf_phot[:, 0])
    sf_color_y = np.log10(sf_phot[:, 3] / sf_phot[:, 1])
# Remove infs/nans
mask_sf = np.isfinite(sf_color_x) & np.isfinite(sf_color_y)
sf_color_x = sf_color_x[mask_sf]
sf_color_y = sf_color_y[mask_sf]

# --- Composite sweep: host → AGN-dominated, fixed z = 0.5 ---
agn_frac_vals = np.array([0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 0.95])
z_agn = 0.5

agn_phot = np.array([composite_photometry(ssp, float(f), z_agn) for f in agn_frac_vals])
agn_color_x = np.log10(agn_phot[:, 2] / agn_phot[:, 0])
agn_color_y = np.log10(agn_phot[:, 3] / agn_phot[:, 1])

# --- Donley+2012 wedge boundaries (Eq. 1-2, simplified) ---
# log(f_5.8/f_3.6) vs log(f_8.0/f_4.5) selection wedge
# Wedge defined by Irac-1 vs Irac-2 color boundaries
# Upper boundary: log(f_8.0/f_4.5) = 0.5 * log(f_5.8/f_3.6) + 0.2
# Lower boundary: log(f_8.0/f_4.5) = 1.0 * log(f_5.8/f_3.6) - 0.3

x_wedge = np.linspace(-0.5, 1.0, 100)
y_upper = 0.5 * x_wedge + 0.2
y_lower = 1.0 * x_wedge - 0.3

# --- Plot ---
fig, ax = plt.subplots(figsize=(7.5, 6.0))

# Wedge polygon
wedge_x = np.concatenate([x_wedge, x_wedge[::-1]])
wedge_y = np.concatenate([y_upper, y_lower[::-1]])
ax.fill(wedge_x, wedge_y, color="red", alpha=0.15, label="AGN wedge (Donley+2012)")

# Wedge boundaries
ax.plot(x_wedge, y_upper, "r--", lw=1.0, alpha=0.6)
ax.plot(x_wedge, y_lower, "r--", lw=1.0, alpha=0.6)

# Star-forming galaxies (only plot finite points)
if len(sf_color_x) > 0:
    ax.scatter(
        sf_color_x,
        sf_color_y,
        s=40,
        alpha=0.6,
        color="C0",
        edgecolors="C0",
        linewidth=0.5,
        label=f"Star-forming galaxies (z=0-2, N={len(sf_color_x)})",
        zorder=3,
    )

# Composite host+AGN track, colored by AGN fraction
scatter_agn = ax.scatter(
    agn_color_x,
    agn_color_y,
    s=120,
    c=agn_frac_vals,
    cmap="autumn_r",
    marker="*",
    edgecolors="darkred",
    linewidth=0.6,
    label=f"Host + AGN composite (z={z_agn})",
    zorder=4,
)
ax.plot(agn_color_x, agn_color_y, color="darkred", lw=0.8, alpha=0.4, zorder=3)
cbar = fig.colorbar(scatter_agn, ax=ax, pad=0.02)
cbar.set_label(r"AGN fraction $f_{\rm AGN}$", fontsize=9)

ax.set_xlabel(r"$\log(f_{5.8} / f_{3.6})$", fontsize=11)
ax.set_ylabel(r"$\log(f_{8.0} / f_{4.5})$", fontsize=11)
ax.set_xlim(-0.6, 1.0)
ax.set_ylim(-0.8, 0.6)
ax.grid(True, alpha=0.3, linestyle=":", linewidth=0.7)
ax.legend(fontsize=9, frameon=False, loc="upper left")

fig.tight_layout()
plt.savefig("plot_spitzer_irac_agn_wedge.png", dpi=150, bbox_inches="tight")

# --- References (docstring doctest-style) ---
#
# .. [1] Lacy M, et al. 2007, ApJ 669, 54–64 (arXiv:0705.4277)
#        "Mid-Infrared Selection of AGN with the Spitzer Space Telescope"
#
# .. [2] Donley JL, et al. 2012, ApJ 748, 142 (arXiv:1202.3816)
#        "Spitzer Quasar and ULIRG Evolution Study (SQUIRES)"